In [1]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cv_a2/synthetic_clips
!nvidia-smi

Mounted at /content/drive
Sun Nov  8 10:06:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg "bitsandbytes>=0.46.1"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import torch, gc
from transformers import UMT5EncoderModel, BitsAndBytesConfig
from diffusers import WanPipeline

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

prompt = ("Close-up of a human hand making a thumbs-up gesture with the thumb pointing sideways, "
          "hand resting on a wooden table, filmed from above, slowly rotating, "
          "soft warm indoor light, realistic, sharp focus")
negative = ("extra fingers, missing fingers, fused fingers, deformed hand, poorly drawn hands, "
            "blurry, low quality, subtitles, text, multiple hands, cartoon")

text_encoder = UMT5EncoderModel.from_pretrained(
    model_id, subfolder="text_encoder",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    torch_dtype=torch.float16, device_map="cuda")

pipe_txt = WanPipeline.from_pretrained(
    model_id, text_encoder=text_encoder, transformer=None, vae=None,
    torch_dtype=torch.float16)

with torch.no_grad():
    prompt_embeds, negative_embeds = pipe_txt.encode_prompt(
        prompt=prompt, negative_prompt=negative,
        do_classifier_free_guidance=True, device="cuda")

prompt_embeds = prompt_embeds.to("cpu", torch.bfloat16)
negative_embeds = negative_embeds.to("cpu", torch.bfloat16)

del pipe_txt, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("Промпт закодирован")

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

Промпт закодирован


In [4]:
import time
from diffusers.utils import export_to_video

pe = prompt_embeds.to("cuda", torch.bfloat16)
ne = negative_embeds.to("cuda", torch.bfloat16)

t0 = time.time()
frames = pipe(prompt_embeds=pe, negative_prompt_embeds=ne,
              height=576, width=320, num_frames=33, guidance_scale=5.0).frames[0]
print(f"Генерация заняла {(time.time()-t0)/60:.1f} мин")

export_to_video(frames, "/content/drive/MyDrive/cv_a2/synthetic_clips/test_thumbs_up_wood.mp4", fps=15)
print("Готово")

  0%|          | 0/50 [00:00<?, ?it/s]

Генерация заняла 19.7 мин
Готово


In [2]:
import torch, gc
from transformers import UMT5EncoderModel, BitsAndBytesConfig
from diffusers import WanPipeline

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

negative = ("extra fingers, missing fingers, fused fingers, deformed hand, poorly drawn hands, "
            "blurry, low quality, subtitles, text, multiple hands, cartoon")

items = [
 ("thumbs_side_wood_warm",  "lying on its side on a table, fingers curled, the thumb pointing horizontally to the left",
                            "wooden table, camera looking straight down from above, warm indoor light"),
 ("thumbs_side_beige_soft", "lying on its side, fingers curled, the thumb pointing horizontally to the right",
                            "beige fabric sofa surface, camera looking down from above, soft light"),
 ("thumbs_side_dark",       "lying on its side, fingers curled, the thumb pointing horizontally to the left",
                            "dark room, dim low light, camera looking down from above"),
 ("thumbs_up_wall_light",   "making a thumbs-up gesture, thumb pointing up, fingers curled",
                            "white marble wall background, side light"),
 ("stop_beige_spread",      "open palm stop gesture, all five fingers spread wide, thumb spread far to the side",
                            "beige fabric surface, camera looking down from above, soft light"),
 ("stop_dark",              "open palm stop gesture, all five fingers spread",
                            "dark room, dim low light"),
 ("peace_dark",             "peace sign V gesture, index and middle finger raised",
                            "dark room, dim low light"),
 ("peace_outdoor",          "peace sign V gesture, index and middle finger raised",
                            "outdoors, green trees in the background, daylight"),
 ("fist_wood_warm",         "closed fist",
                            "wooden table, camera looking down from above, warm indoor light"),
 ("fist_outdoor",           "closed fist",
                            "outdoors, street background, daylight"),
]

text_encoder = UMT5EncoderModel.from_pretrained(
    model_id, subfolder="text_encoder",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    torch_dtype=torch.float16, device_map="cuda")
pipe_txt = WanPipeline.from_pretrained(
    model_id, text_encoder=text_encoder, transformer=None, vae=None, torch_dtype=torch.float16)

embeds = {}
with torch.no_grad():
    for name, pose, scene in items:
        prompt = f"Close-up of a human hand, {pose}, {scene}, slowly rotating, realistic, sharp focus"
        pe, ne = pipe_txt.encode_prompt(prompt=prompt, negative_prompt=negative,
                                        do_classifier_free_guidance=True, device="cuda")
        embeds[name] = (pe.to("cpu", torch.bfloat16), ne.to("cpu", torch.bfloat16))

del pipe_txt, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("Закодировано промптов:", len(embeds))

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

Закодировано промптов: 10


In [3]:
import os, time
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

OUT = "/content/drive/MyDrive/cv_a2/synthetic_clips"
SEEDS = [0]
USE_OFFLOAD = False

vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, text_encoder=None, tokenizer=None,
                                   torch_dtype=torch.bfloat16)
if USE_OFFLOAD:
    pipe.enable_model_cpu_offload()
else:
    pipe.to("cuda")
pipe.vae.enable_tiling()

for name, (pe, ne) in embeds.items():
    for seed in SEEDS:
        path = f"{OUT}/{name}_s{seed}.mp4"
        if os.path.exists(path):
            print("пропуск", path); continue
        t0 = time.time()
        frames = pipe(prompt_embeds=pe.to("cuda"), negative_prompt_embeds=ne.to("cuda"),
                      height=576, width=320, num_frames=33, num_inference_steps=25,
                      guidance_scale=5.0, generator=torch.Generator("cpu").manual_seed(seed)).frames[0]
        export_to_video(frames, path, fps=15)
        print(f"{name} s{seed}: {(time.time()-t0)/60:.1f} мин")
print("Всё готово")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

thumbs_side_wood_warm s0: 10.1 мин


  0%|          | 0/25 [00:00<?, ?it/s]

... (13 outputs left out)

  0%|          | 0/25 [00:00<?, ?it/s]

fist_wood_warm s0: 10.0 мин


  0%|          | 0/25 [00:00<?, ?it/s]

fist_outdoor s0: 10.0 мин
Всё готово
